# Database initialization

In [ ]:
from db_management_tools import DataLoader
dataset_dir = r"D:\AIC26"
data_loader = DataLoader(dataset_dir)

# Upload data

In [1]:
data_loader.create_and_fill_database(
    "aic26_mini",
    upload_batch_size=1024,
    upload_num_workers=8
)

NameError: name 'data_loader' is not defined

# Query initialization

In [2]:
from db_management_tools import QueryManager
from models import SigLIP2Model, BGEM3Model
from peripheral_tools import NotebookDisplay

image_text_feature_extractor = SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda")
text_text_feature_extractor = BGEM3Model(name="BAAI/bge-m3", device="cuda")

query_manager = QueryManager(
    image_text_feature_extractor=image_text_feature_extractor,
    text_text_feature_extractor=text_text_feature_extractor
)
display = NotebookDisplay(315, 5)

[INFO] Loading quantized google/siglip2-so400m-patch14-384
[INFO] google/siglip2-so400m-patch14-384 is on cuda device


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [4]:
info = query_manager.describe_fields('aic26_mini')

In [9]:
info.config

CollectionConfig(params=CollectionParams(vectors={'beit3_dense': VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=True, datatype=None, multivector_config=None), 'bgem3_dense': VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=True, datatype=None, multivector_config=None), 'siglip2_dense': VectorParams(size=1152, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=True, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors={'bgem3_sparse': SparseVectorParams(index=SparseIndexParams(full_scan_threshold=None, on_disk=True, datatype=None), modifier=<Modifier.IDF: 'idf'>)}), hnsw_config=HnswConfig(m=32, ef_construct=200, full_scan_threshold=10000, max_indexing_th

# Query test

In [ ]:
query = input("Enter your query: ")
limit = int(input("Enter your search limit: "))
points = query_manager.search(
    "aic26_mini",
    query=query,
    search_limit=limit,
    using_vectors=["siglip2_dense"]
)

display.display(points)

# Benchmark

In [4]:
from peripheral_tools import Benchmarker, BenchmarkerV2

test_settings = [
    {
        "using_vectors": ["siglip2_dense"]
    },
    {
        "using_vectors": ["siglip2_dense", "bgem3_sparse"]
    },
    {
        "using_vectors": ["siglip2_dense", "bgem3_dense"]
    },
    {
        "using_vectors": ["siglip2_dense", "bgem3_dense", "bgem3_sparse"]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.4,
        "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.5,
        "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.4,
        "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.5,
        "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.4,
        "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.5,
        "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    },
]

benchmarker = BenchmarkerV2(query_manager)
res = benchmarker.do_benchmark(
    "aic26_mini",
    test_questions=r"C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\GITHUB_REPOS\Determinative-HCMCAIC26\question_bank\questions_with_answers.csv",
    test_settings=test_settings,
    save_record=False
)

Doing benchmark: 100%|██████████| 44/44 [10:29<00:00, 14.30s/it]

Benchmark results:
           Result@1  Result@5  Result@10  Result@50  mean_MRR  median_MRR  min_MRR  max_MRR   std_MRR  mean_Runtime  median_Runtime  min_Runtime  max_Runtime  std_Runtime
Method 1   0.431818  0.727273   0.772727   0.840909  0.567323         0.5      0.0      1.0  0.410614      0.032017        0.031786     0.019172     0.090646     0.011545
Method 2   0.386364  0.886364   0.909091   0.931818  0.587546         0.5      0.0      1.0  0.352712      0.059459        0.053894     0.036184     0.361591     0.046751
Method 3   0.522727  0.840909   0.886364   0.931818  0.658972         1.0      0.0      1.0  0.380471      0.052751        0.051963     0.038894     0.082170     0.010652
Method 4   0.568182  0.886364   0.931818   0.954545  0.691225         1.0      0.0      1.0  0.370326      0.053502        0.055019     0.037954     0.069736     0.007741
Method 5   0.613636  0.909091   0.931818   0.954545  0.725325         1.0      0.0      1.0  0.361662      2.723141        2.6